# Step 2 (Colab) — Train ACT, with checkpoints saved to Google Drive

Colab-ready version of `act_training.ipynb`. Loads the LeRobot dataset produced by `convert_alllog_to_lerobot_colab.ipynb` from Drive and writes checkpoints back to Drive so they survive runtime resets.

**Before running**

1. Set the Colab runtime to a GPU (Runtime → Change runtime type → T4 or better).
2. Make sure `convert_alllog_to_lerobot_colab.ipynb` finished — the dataset must exist at `MyDrive/Lebai_train_ACT/lerobot_cache/local/lebai_duck_pick/`.

## 1. Install dependencies

In [ ]:
!pip install -q lerobot matplotlib

## 2. Mount Drive and configure paths

Set `HF_LEROBOT_HOME` to the same Drive directory the converter wrote to, **before** importing `lerobot`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Lebai_train_ACT')
LEROBOT_CACHE  = DRIVE_ROOT / 'lerobot_cache'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/act_run01'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_LEROBOT_HOME'] = str(LEROBOT_CACHE)

assert LEROBOT_CACHE.exists(), f'No converted dataset at {LEROBOT_CACHE} — run the converter notebook first.'
print(f'HF_LEROBOT_HOME = {os.environ["HF_LEROBOT_HOME"]}')
print(f'CHECKPOINT_DIR  = {CHECKPOINT_DIR}')

## 3. Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from lerobot.common.policies.act.modeling_act import ACTPolicy
from lerobot.common.policies.act.configuration_act import ACTConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no GPU — training will be very slow. Runtime → Change runtime type → GPU.')

## 4. Load dataset metadata

In [ ]:
DATASET_REPO_ID = 'local/lebai_duck_pick'  # must match REPO_ID from the converter

dataset_meta = LeRobotDataset(DATASET_REPO_ID).meta
print(f'  episodes: {dataset_meta.total_episodes}')
print(f'  frames:   {dataset_meta.total_frames}')
print(f'  fps:      {dataset_meta.fps}')
print(f'  tasks:    {len(dataset_meta.tasks)}')
print('Features:')
for name, spec in dataset_meta.features.items():
    print(f'  {name}: dtype={spec["dtype"]}, shape={spec.get("shape")}')

fps = dataset_meta.fps

## 5. Sanity-check the dataset

Single highest-ROI check in the pipeline — converter bugs (wrong channel, swapped joints, action == state) are nearly invisible from training loss alone but obvious here.

In [ ]:
ds_inspect = LeRobotDataset(DATASET_REPO_ID)
sample = ds_inspect[0]
image_keys = [k for k in sample.keys() if k.startswith('observation.images.')]

fig, axes = plt.subplots(1, len(image_keys), figsize=(5 * len(image_keys), 5))
if len(image_keys) == 1:
    axes = [axes]
for ax, key in zip(axes, image_keys):
    img = sample[key].permute(1, 2, 0).cpu().numpy()
    ax.imshow(img); ax.set_title(key.replace('observation.images.', '')); ax.axis('off')
plt.tight_layout(); plt.show()

print(f"state:  {sample['observation.state'].numpy().round(3)}")
print(f"action: {sample['action'].numpy().round(3)}")
print(f"task:   {sample['task']!r}")

In [ ]:
ep0_from = ds_inspect.episode_data_index['from'][0].item()
ep0_to   = ds_inspect.episode_data_index['to'][0].item()
frames = [ds_inspect[i] for i in range(ep0_from, ep0_to)]

states  = torch.stack([f['observation.state'] for f in frames]).numpy()
actions = torch.stack([f['action'] for f in frames]).numpy()
t = np.arange(len(states)) / fps
arm_dims = states.shape[1] - 1

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for i in range(arm_dims):
    axes[0, 0].plot(t, states[:, i], label=f'j{i}')
    axes[0, 1].plot(t, actions[:, i], label=f'j{i}')
axes[1, 0].plot(t, states[:, -1], color='black', label='gripper')
axes[1, 1].plot(t, actions[:, -1], color='black', label='gripper')
axes[0, 0].set_title('state — arm joints'); axes[0, 0].legend(loc='upper right')
axes[0, 1].set_title('action — arm joints'); axes[0, 1].legend(loc='upper right')
axes[1, 0].set_title('state — gripper amplitude'); axes[1, 0].set_xlabel('time (s)')
axes[1, 1].set_title('action — gripper amplitude'); axes[1, 1].set_xlabel('time (s)')
for a in axes.flat: a.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Configure ACT

`chunk_size=32` ≈ 3.2 s of lookahead at 10 Hz (ALOHA defaults assume 50 Hz). `n_decoder_layers=1` matches the original ACT implementation.

In [ ]:
CHUNK_SIZE = 32
N_ACTION_STEPS = CHUNK_SIZE

delta_timestamps = {'action': [t / fps for t in range(CHUNK_SIZE)]}
dataset = LeRobotDataset(DATASET_REPO_ID, delta_timestamps=delta_timestamps)
print(f'Training frames: {len(dataset)} (each yields a chunk of {CHUNK_SIZE} actions)')

cfg = ACTConfig(
    n_obs_steps=1,
    chunk_size=CHUNK_SIZE,
    n_action_steps=N_ACTION_STEPS,
    vision_backbone='resnet18',
    pretrained_backbone_weights='ResNet18_Weights.IMAGENET1K_V1',
    replace_final_stride_with_dilation=False,
    dim_model=512,
    n_heads=8,
    dim_feedforward=3200,
    n_encoder_layers=4,
    n_decoder_layers=1,
    use_vae=True,
    latent_dim=32,
    n_vae_encoder_layers=4,
    kl_weight=10.0,
    dropout=0.1,
)
cfg.input_features  = {k: v for k, v in dataset.features.items() if k.startswith('observation.')}
cfg.output_features = {'action': dataset.features['action']}

policy = ACTPolicy(cfg, dataset_stats=dataset.meta.stats)
policy.to(device); policy.train()

n_params = sum(p.numel() for p in policy.parameters())
print(f'Policy: {n_params/1e6:.1f}M parameters')

## 7. Optionally resume from a Drive checkpoint

Colab can disconnect after a few hours. This cell finds the most recent `step_*` checkpoint under `CHECKPOINT_DIR` and loads it. Skip if you want to start fresh.

In [ ]:
RESUME = True

start_step = 0
if RESUME:
    ckpts = sorted(CHECKPOINT_DIR.glob('step_*'))
    if ckpts:
        latest = ckpts[-1]
        print(f'Resuming from {latest}')
        policy = ACTPolicy.from_pretrained(latest)
        policy.to(device); policy.train()
        start_step = int(latest.name.split('_')[1])
        print(f'Resumed at step {start_step}')
    else:
        print('No checkpoint found — training from scratch.')
else:
    print('RESUME=False — training from scratch.')

## 8. Training loop

In [ ]:
BATCH_SIZE = 8         # raise to 16/32 on A100
NUM_STEPS  = 50_000
LOG_EVERY  = 200
SAVE_EVERY = 5_000     # writes go to Drive — keeps progress across runtime resets

dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=device.type == 'cuda', drop_last=True,
)
def cycle(loader):
    while True:
        for batch in loader:
            yield batch
step_iter = cycle(dataloader)

optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_STEPS)
for _ in range(start_step):
    scheduler.step()

print(f'Will run steps {start_step} → {NUM_STEPS}  bs={BATCH_SIZE}  -> {CHECKPOINT_DIR}')

In [ ]:
loss_history = []
t0 = time.time()

for step in range(start_step, NUM_STEPS):
    batch = next(step_iter)
    batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
             for k, v in batch.items()}

    output_dict = policy.forward(batch)
    loss = output_dict['loss']

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    loss_history.append(loss.item())

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        rate = (step - start_step + 1) / max(elapsed, 1e-6)
        print(f'step {step:6d}  loss={loss.item():.4f}  lr={scheduler.get_last_lr()[0]:.2e}  ({rate:.1f} step/s)')

    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = CHECKPOINT_DIR / f'step_{step+1:06d}'
        policy.save_pretrained(ckpt_path)
        print(f'  -> saved {ckpt_path}')

final_ckpt = CHECKPOINT_DIR / 'final'
policy.save_pretrained(final_ckpt)
print(f'\nDone. Final: {final_ckpt}')

## 9. Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_history, alpha=0.4, label='raw')
window = max(1, len(loss_history) // 100)
if len(loss_history) >= window:
    smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(loss_history)), smoothed, label=f'smoothed (w={window})')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title('ACT training loss'); plt.tight_layout(); plt.show()

## Next

`CHECKPOINT_DIR/final/` is persisted to Drive. Download it (or copy to the robot machine) and point `act_inference.ipynb`'s `CHECKPOINT_PATH` at it. Inference cannot run on Colab — it needs LAN access to the Lebai SDK and the camera service.